# Auxiliary Data Preprocessing and Merge

This notebook adds the selected auxiliary features to the deduplicated core tables.

The final `core + hdb_mrt` datasets include:

- HDB block coordinates and attributes;
- distance and counts for a conservative pre-2021 MRT network proxy.

The supplied MRT status is a current snapshot, so it is not used as evidence that a station existed in every rental month. Known post-2021 and future codes are excluded. School, mall, COE, and stock tables remain separate ablation artifacts.

After the merge, duplicate processed feature vectors are audited. Training vectors that are identical across every model input are collapsed to one row and their rent is averaged, which matches the squared-error objective. Test rows are never collapsed because every submission ID needs a prediction.


In [1]:
from pathlib import Path
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.neighbors import BallTree

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

EARTH_RADIUS_KM = 6_371.0088
SAVE_OUTPUTS = True


def locate_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "data/train.csv").exists() and (candidate / "data/auxiliary").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository data directory")


ROOT = locate_project_root()
DATA_DIR = ROOT / "data"
AUX_DIR = DATA_DIR / "auxiliary"
PROCESSED_DIR = DATA_DIR / "processed"
CORE_DIR = PROCESSED_DIR / "core"
HDB_MRT_DIR = PROCESSED_DIR / "hdb_mrt"
AUXILIARY_OUTPUT_DIR = PROCESSED_DIR / "auxiliary"

paths = {
    "train_raw": DATA_DIR / "train.csv",
    "test_raw": DATA_DIR / "test.csv",
    "train_processed": CORE_DIR / "train_preprocessed.csv",
    "test_processed": CORE_DIR / "test_preprocessed.csv",
    "processed_metadata": CORE_DIR / "preprocessing_metadata.json",
    "hdb": AUX_DIR / "sg-hdb-block.csv",
    "mrt": AUX_DIR / "sg-mrt-stations.csv",
    "schools": AUX_DIR / "sg-schools.csv",
    "malls": AUX_DIR / "sg-shopping-malls.csv",
    "stocks": AUX_DIR / "sg-stock-prices.csv",
    "coe": AUX_DIR / "sg-coe-prices.csv",
}
missing_files = [str(input_path) for input_path in paths.values() if not input_path.exists()]
assert not missing_files, f"Missing input files: {missing_files}"

print(f"Project root: {ROOT}")
print("Final merge policy: deduplicated core + HDB + conservative pre-2021 MRT proxy")

Project root: /Users/ziqing/cs5228/cs5228-group6
Final merge policy: deduplicated core + HDB + conservative pre-2021 MRT proxy


## 1. Load and audit source tables

Identifiers such as postal codes are loaded as strings. The audit is descriptive and does not modify the sources.

In [2]:
train_raw = pd.read_csv(paths["train_raw"])
test_raw = pd.read_csv(paths["test_raw"])
train_processed = pd.read_csv(paths["train_processed"], low_memory=False)
test_processed = pd.read_csv(paths["test_processed"], low_memory=False)
with paths["processed_metadata"].open(encoding="utf-8") as handle:
    processed_metadata = json.load(handle)

hdb_raw = pd.read_csv(paths["hdb"], dtype={"POSTAL_CODE": "string"})
mrt_raw = pd.read_csv(paths["mrt"])
schools_raw = pd.read_csv(paths["schools"], dtype={"POSTAL_CODE": "string"})
malls_raw = pd.read_csv(paths["malls"])
stocks_raw = pd.read_csv(paths["stocks"])
coe_raw = pd.read_csv(paths["coe"])

frames = {
    "train_raw": train_raw,
    "test_raw": test_raw,
    "train_processed": train_processed,
    "test_processed": test_processed,
    "hdb": hdb_raw,
    "mrt": mrt_raw,
    "schools": schools_raw,
    "malls": malls_raw,
    "stocks": stocks_raw,
    "coe": coe_raw,
}

source_audit = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "duplicate_rows": int(frame.duplicated().sum()),
        "missing_cells": int(frame.isna().sum().sum()),
    }
    for name, frame in frames.items()
]).set_index("dataset")
display(source_audit)

assert "MONTHLY_RENT" in train_processed.columns
assert "MONTHLY_RENT" not in test_processed.columns
assert len(train_processed) <= len(train_raw)
assert set(train_processed["ROW_ID"]).issubset(set(range(len(train_raw))))
assert len(test_processed) == len(test_raw)
assert train_processed["ROW_ID"].is_unique and test_processed["ROW_ID"].is_unique

,rows,columns,duplicate_rows,missing_cells
dataset,,,,
train_raw,150000,11,425,0
test_raw,50000,10,5508,0
train_processed,149359,22,0,1
test_processed,50000,21,0,0
hdb,9464,11,0,0
mrt,243,8,0,0
schools,337,14,0,0
malls,113,3,0,0
stocks,91642,9,0,510


## 2. Shared normalization helpers

Address keys use Unicode normalization, lowercase text, trimmed whitespace, and collapsed internal spaces. The same functions are applied to HDB, train, and test records.

In [3]:
ADDRESS_KEYS = ["TOWN", "BLOCK", "STREET"]


def normalize_value(value):
    if pd.isna(value):
        return pd.NA
    value = unicodedata.normalize("NFKC", str(value)).strip().lower()
    return re.sub(r"\s+", " ", value)


def normalize_columns(frame, columns):
    result = frame.copy()
    for column in columns:
        result[column] = result[column].map(normalize_value).astype("string")
    return result


def parse_rent_month(values):
    text = values.astype("string").str.strip()
    parsed = pd.Series(pd.NaT, index=values.index, dtype="datetime64[ns]")
    for date_format in ("%Y-%m", "%b-%y"):
        missing = parsed.isna()
        parsed.loc[missing] = pd.to_datetime(text.loc[missing], format=date_format, errors="coerce")
    if parsed.isna().any():
        bad = text.loc[parsed.isna()].drop_duplicates().tolist()[:10]
        raise ValueError(f"Unrecognized rental month values: {bad}")
    return parsed.dt.to_period("M").dt.to_timestamp()


def clean_number(values):
    return pd.to_numeric(
        values.astype("string").str.replace(r"[$,]", "", regex=True).str.strip(),
        errors="coerce",
    )


def validate_coordinates(frame):
    latitude = pd.to_numeric(frame["LATITUDE"], errors="coerce")
    longitude = pd.to_numeric(frame["LONGITUDE"], errors="coerce")
    valid = latitude.between(-90, 90) & longitude.between(-180, 180)
    result = frame.loc[valid].copy()
    result["LATITUDE"] = latitude.loc[valid]
    result["LONGITUDE"] = longitude.loc[valid]
    return result

## 3. HDB block metadata

The investigation found complete address coverage and valid coordinates for all 9,464 blocks. Sentinel values such as `-1` and `NIL` are converted to missing values and accompanied by explicit quality flags.

In [4]:
hdb = normalize_columns(hdb_raw, ADDRESS_KEYS)
hdb = validate_coordinates(hdb)

duplicate_hdb_keys = int(hdb.duplicated(ADDRESS_KEYS).sum())
if duplicate_hdb_keys:
    conflicting = hdb.groupby(ADDRESS_KEYS, dropna=False).nunique(dropna=False).max(axis=1).gt(1)
    assert not conflicting.any(), "Conflicting rows found for the same HDB address key"
    hdb = hdb.drop_duplicates(ADDRESS_KEYS, keep="first")

hdb["MAX_FLOOR"] = pd.to_numeric(hdb["MAX_FLOOR"], errors="coerce")
hdb["YEAR_COMPLETED"] = pd.to_numeric(hdb["YEAR_COMPLETED"], errors="coerce")
hdb["MAX_FLOOR_INVALID"] = hdb["MAX_FLOOR"].le(0).fillna(True).astype("int8")
hdb["YEAR_COMPLETED_INVALID"] = hdb["YEAR_COMPLETED"].le(0).fillna(True).astype("int8")
hdb.loc[hdb["MAX_FLOOR"].le(0), "MAX_FLOOR"] = np.nan
hdb.loc[hdb["YEAR_COMPLETED"].le(0), "YEAR_COMPLETED"] = np.nan

hdb["POSTAL_CODE"] = hdb["POSTAL_CODE"].str.strip().replace({"NIL": pd.NA, "nil": pd.NA})
hdb["POSTAL_CODE"] = hdb["POSTAL_CODE"].str.zfill(6)
hdb["POSTAL_SECTOR"] = hdb["POSTAL_CODE"].str[:2]
for column in ["SUBZONE", "PLANNING_AREA", "REGION"]:
    hdb[column] = hdb[column].map(normalize_value).astype("string")

hdb = hdb.rename(columns={
    "POSTAL_CODE": "HDB_POSTAL_CODE",
    "POSTAL_SECTOR": "HDB_POSTAL_SECTOR",
    "LATITUDE": "HDB_LATITUDE",
    "LONGITUDE": "HDB_LONGITUDE",
    "MAX_FLOOR": "HDB_MAX_FLOOR",
    "YEAR_COMPLETED": "HDB_YEAR_COMPLETED",
    "MAX_FLOOR_INVALID": "HDB_MAX_FLOOR_INVALID",
    "YEAR_COMPLETED_INVALID": "HDB_YEAR_COMPLETED_INVALID",
    "SUBZONE": "HDB_SUBZONE",
    "PLANNING_AREA": "HDB_PLANNING_AREA",
    "REGION": "HDB_REGION",
})

assert not hdb.duplicated(ADDRESS_KEYS).any()
assert hdb[["HDB_LATITUDE", "HDB_LONGITUDE"]].notna().all().all()

display(pd.Series({
    "unique_blocks": len(hdb),
    "duplicate_keys_removed": duplicate_hdb_keys,
    "missing_postal_codes": int(hdb["HDB_POSTAL_CODE"].isna().sum()),
    "invalid_max_floor": int(hdb["HDB_MAX_FLOOR_INVALID"].sum()),
    "invalid_year_completed": int(hdb["HDB_YEAR_COMPLETED_INVALID"].sum()),
}, name="HDB preprocessing"))

unique_blocks             9464
duplicate_keys_removed       0
missing_postal_codes       285
invalid_max_floor           11
invalid_year_completed      11
Name: HDB preprocessing, dtype: int64

## 4. MRT, schools, and shopping malls

Duplicate coordinates are removed before spatial queries. For MRT, the current `STATUS` field is combined with a conservative line/code allowlist that approximates the network available at the start of training in January 2021. The proxy deliberately excludes the Thomson-East Coast Line and known later infill or extension codes. This reduces historical leakage but does not replace authoritative station opening dates.


In [5]:
def clean_point_table(frame):
    result = validate_coordinates(frame.copy())
    result["NAME"] = result["NAME"].map(normalize_value).astype("string")
    return result.drop_duplicates(["NAME", "LATITUDE", "LONGITUDE"]).reset_index(drop=True)


PRE2021_MRT_PREFIXES = {"ns", "ew", "cg", "ne", "cc", "ce", "dt"}
POST2021_OR_FUTURE_CODES = {"ns6", "ne18", "cc30", "cc31", "cc32", "dt4", "dt36", "dt37"}


def conservative_pre2021_mrt(frame):
    result = frame.copy()
    result["CODE_CLEAN"] = result["CODE"].map(normalize_value)
    result["CODE_PREFIX"] = result["CODE_CLEAN"].str.extract(r"^([a-z]+)", expand=False)
    status = result["STATUS"].map(normalize_value)
    result = result.loc[
        status.eq("open")
        & result["CODE_PREFIX"].isin(PRE2021_MRT_PREFIXES)
        & ~result["CODE_CLEAN"].isin(POST2021_OR_FUTURE_CODES)
    ].copy()
    return clean_point_table(result)


mrt = conservative_pre2021_mrt(mrt_raw)
schools = clean_point_table(schools_raw)
malls = clean_point_table(malls_raw)

primary_mask = schools["MAINLEVEL_CODE"].astype("string").str.lower().str.contains(
    r"primary|p1", regex=True, na=False
)
primary_schools = schools.loc[primary_mask].copy()

amenity_inventory = pd.DataFrame({
    "dataset": ["mrt_pre2021_proxy", "school", "primary_school", "mall"],
    "usable_locations": [len(mrt), len(schools), len(primary_schools), len(malls)],
}).set_index("dataset")
display(amenity_inventory)


,usable_locations
dataset,
mrt_pre2021_proxy,122
school,337
primary_school,182
mall,113


In [6]:
def spatial_features(origins, destinations, prefix):
    origin_coordinates = np.radians(
        origins[["HDB_LATITUDE", "HDB_LONGITUDE"]].to_numpy(dtype=float)
    )
    destination_coordinates = np.radians(
        destinations[["LATITUDE", "LONGITUDE"]].to_numpy(dtype=float)
    )
    tree = BallTree(destination_coordinates, metric="haversine")
    nearest_radians, _ = tree.query(origin_coordinates, k=1)
    return pd.DataFrame({
        f"DIST_TO_NEAREST_{prefix}_KM": (nearest_radians[:, 0] * EARTH_RADIUS_KM).astype("float32"),
        f"{prefix}_COUNT_WITHIN_500M": tree.query_radius(
            origin_coordinates, r=0.5 / EARTH_RADIUS_KM, count_only=True
        ).astype("int16"),
        f"{prefix}_COUNT_WITHIN_1KM": tree.query_radius(
            origin_coordinates, r=1.0 / EARTH_RADIUS_KM, count_only=True
        ).astype("int16"),
    }, index=origins.index)


block_features = hdb.copy()
for prefix, destinations in {
    "PRE2021_MRT": mrt,
    "SCHOOL": schools,
    "PRIMARY_SCHOOL": primary_schools,
    "MALL": malls,
}.items():
    block_features = pd.concat(
        [block_features, spatial_features(block_features, destinations, prefix)],
        axis=1,
    )

distance_columns = [column for column in block_features if column.startswith("DIST_TO_NEAREST_")]
count_columns = [column for column in block_features if "_COUNT_WITHIN_" in column]
assert block_features[distance_columns + count_columns].notna().all().all()

display(block_features[distance_columns].describe(percentiles=[0.5, 0.9]).T[["mean", "50%", "90%", "max"]])

,mean,50%,90%,max
DIST_TO_NEAREST_PRE2021_MRT_KM,0.7667,0.6883,1.3446,3.5147
DIST_TO_NEAREST_SCHOOL_KM,0.3334,0.2884,0.5756,3.3092
DIST_TO_NEAREST_PRIMARY_SCHOOL_KM,0.4129,0.3664,0.7104,3.3092
DIST_TO_NEAREST_MALL_KM,0.8290,0.7473,1.5216,3.2024


### MRT time-safety decision

The current status snapshot cannot establish whether a station was open in an earlier month. The output therefore uses a fixed pre-2021 proxy across train and test. This avoids treating later openings as historical amenities. A stronger future version should join authoritative opening dates and recompute proximity for each rental month.


## 5. Optional COE features

COE fields are cleaned by category and month. Auction rounds are aggregated, and the resulting table is shifted by one calendar month before it can be joined to rental records.

In [7]:
coe = coe_raw.copy()
coe["CATEGORY"] = coe["CATEGORY"].map(normalize_value).str.upper()
coe["MONTH"] = pd.to_datetime(
    coe["YEAR"].astype(str) + "-" + coe["MONTH"].astype(str),
    format="%Y-%b",
    errors="raise",
).dt.to_period("M").dt.to_timestamp()

for column in ["QUOTA_PREMIUM", "PREVAILING_QUOTA_PREMIUM", "QUOTA", "BIDS_RECEIVED"]:
    coe[column] = clean_number(coe[column])

coe_monthly_long = coe.groupby(["MONTH", "CATEGORY"], as_index=False).agg(
    PREMIUM_MEAN=("QUOTA_PREMIUM", "mean"),
    PREVAILING_PREMIUM_MEAN=("PREVAILING_QUOTA_PREMIUM", "mean"),
    QUOTA_SUM=("QUOTA", "sum"),
    BIDS_SUM=("BIDS_RECEIVED", "sum"),
)
coe_monthly_long["BID_TO_QUOTA_RATIO"] = (
    coe_monthly_long["BIDS_SUM"] / coe_monthly_long["QUOTA_SUM"].replace(0, np.nan)
)

coe_monthly = coe_monthly_long.pivot(index="MONTH", columns="CATEGORY")
coe_monthly.columns = [f"COE_{category}_{metric}" for metric, category in coe_monthly.columns]
coe_monthly = coe_monthly.sort_index()
coe_monthly = coe_monthly.reindex(
    pd.date_range(coe_monthly.index.min(), coe_monthly.index.max(), freq="MS")
).shift(1)
coe_monthly.columns = [f"{column}_LAG1" for column in coe_monthly.columns]
coe_monthly.index.name = "RENT_DATE"
coe_monthly = coe_monthly.reset_index()

display(coe_monthly.tail(3))

,RENT_DATE,COE_A_PREMIUM_MEAN_LAG1,COE_B_PREMIUM_MEAN_LAG1,COE_C_PREMIUM_MEAN_LAG1,COE_E_PREMIUM_MEAN_LAG1,COE_A_PREVAILING_PREMIUM_MEAN_LAG1,COE_B_PREVAILING_PREMIUM_MEAN_LAG1,COE_C_PREVAILING_PREMIUM_MEAN_LAG1,COE_E_PREVAILING_PREMIUM_MEAN_LAG1,COE_A_QUOTA_SUM_LAG1,COE_B_QUOTA_SUM_LAG1,COE_C_QUOTA_SUM_LAG1,COE_E_QUOTA_SUM_LAG1,COE_A_BIDS_SUM_LAG1,COE_B_BIDS_SUM_LAG1,COE_C_BIDS_SUM_LAG1,COE_E_BIDS_SUM_LAG1,COE_A_BID_TO_QUOTA_RATIO_LAG1,COE_B_BID_TO_QUOTA_RATIO_LAG1,COE_C_BID_TO_QUOTA_RATIO_LAG1,COE_E_BID_TO_QUOTA_RATIO_LAG1
65,2026-06-01,"124,509.5000","127,868.5000","89,851.0000","128,850.0000","112,324.0000","114,577.0000","77,884.0000",0.0000,2540,1752,585,510,4354,2801,979,915,1.7142,1.5987,1.6735,1.7941
66,2026-07-01,"124,928.0000","125,245.5000","93,500.5000","129,001.0000","118,357.0000","121,218.0000","82,868.0000",0.0000,2497,1772,584,523,3844,2502,843,984,1.5394,1.4120,1.4435,1.8815
67,2026-08-01,"127,500.0000","130,389.5000","94,444.5000","129,886.0000","123,315.0000","124,705.0000","88,368.0000",0.0000,2489,1738,597,540,3469,2865,872,902,1.3937,1.6484,1.4606,1.6704


## 6. Optional stock-market features

Raw price levels are not averaged across companies. Adjusted-close returns are calculated per symbol and summarized robustly by month. Daily changes above 50% in absolute value are excluded from the volatility calculation as likely corporate-action or data-quality artifacts. The monthly table is shifted by one month.

In [8]:
stocks = stocks_raw.copy()
stocks["DATE"] = pd.to_datetime(stocks["DATE"], errors="coerce")
stocks["SYMBOL"] = stocks["SYMBOL"].astype("string")
stocks["ADJUSTED_CLOSE"] = pd.to_numeric(stocks["ADJUSTED_CLOSE"], errors="coerce")
stocks = stocks.dropna(subset=["DATE", "SYMBOL", "ADJUSTED_CLOSE"])
stocks = stocks.loc[stocks["ADJUSTED_CLOSE"].gt(0)].sort_values(["SYMBOL", "DATE"]).copy()
stocks["MONTH"] = stocks["DATE"].dt.to_period("M").dt.to_timestamp()

stocks["DAILY_RETURN"] = stocks.groupby("SYMBOL", observed=True)["ADJUSTED_CLOSE"].pct_change(fill_method=None)
stocks["DAILY_RETURN_CLEAN"] = stocks["DAILY_RETURN"].mask(stocks["DAILY_RETURN"].abs().gt(0.50))
stocks["NORMALIZED_LEVEL"] = (
    stocks["ADJUSTED_CLOSE"] /
    stocks.groupby("SYMBOL", observed=True)["ADJUSTED_CLOSE"].transform("first")
)

daily_volatility = stocks.groupby(["SYMBOL", "MONTH"], observed=True)["DAILY_RETURN_CLEAN"].std().rename("DAILY_VOLATILITY")
stock_month_end = stocks.groupby(["SYMBOL", "MONTH"], observed=True).tail(1).copy()
stock_month_end["MONTHLY_RETURN"] = stock_month_end.groupby("SYMBOL", observed=True)["ADJUSTED_CLOSE"].pct_change(fill_method=None)
stock_month_end = stock_month_end.merge(
    daily_volatility.reset_index(), on=["SYMBOL", "MONTH"], how="left", validate="one_to_one"
)


def positive_share(values):
    valid = values.dropna()
    return valid.gt(0).mean() if len(valid) else np.nan


stock_monthly = stock_month_end.groupby("MONTH", as_index=True, observed=True).agg(
    STOCK_INDEX_LEVEL_MEDIAN=("NORMALIZED_LEVEL", "median"),
    STOCK_RETURN_MEDIAN=("MONTHLY_RETURN", "median"),
    STOCK_RETURN_Q25=("MONTHLY_RETURN", lambda values: values.quantile(0.25)),
    STOCK_RETURN_Q75=("MONTHLY_RETURN", lambda values: values.quantile(0.75)),
    STOCK_POSITIVE_SHARE=("MONTHLY_RETURN", positive_share),
    STOCK_DAILY_VOLATILITY_MEDIAN=("DAILY_VOLATILITY", "median"),
    STOCK_ACTIVE_SYMBOL_COUNT=("SYMBOL", "nunique"),
    STOCK_RETURN_SYMBOL_COUNT=("MONTHLY_RETURN", "count"),
)
stock_monthly["STOCK_RETURN_IQR"] = stock_monthly["STOCK_RETURN_Q75"] - stock_monthly["STOCK_RETURN_Q25"]
stock_monthly = stock_monthly.drop(columns=["STOCK_RETURN_Q25", "STOCK_RETURN_Q75"])
stock_monthly = stock_monthly.reindex(
    pd.date_range(stock_monthly.index.min(), stock_monthly.index.max(), freq="MS")
).shift(1)
stock_monthly.columns = [f"{column}_LAG1" for column in stock_monthly.columns]
stock_monthly.index.name = "RENT_DATE"
stock_monthly = stock_monthly.reset_index()

display(stock_monthly.tail(3))

,RENT_DATE,STOCK_INDEX_LEVEL_MEDIAN_LAG1,STOCK_RETURN_MEDIAN_LAG1,STOCK_POSITIVE_SHARE_LAG1,STOCK_DAILY_VOLATILITY_MEDIAN_LAG1,STOCK_ACTIVE_SYMBOL_COUNT_LAG1,STOCK_RETURN_SYMBOL_COUNT_LAG1,STOCK_RETURN_IQR_LAG1
64,2026-05-01,1.0965,0.0250,0.6623,0.0191,77.0000,77.0000,0.0991
65,2026-06-01,1.0619,0.0000,0.4805,0.0228,77.0000,77.0000,0.1255
66,2026-07-01,1.0720,-0.0201,0.3506,0.0172,77.0000,77.0000,0.1532


In [9]:
market_features = coe_monthly.merge(
    stock_monthly, on="RENT_DATE", how="outer", validate="one_to_one"
).sort_values("RENT_DATE").reset_index(drop=True)

coe_feature_columns = [column for column in market_features if column.startswith("COE_")]
stock_feature_columns = [column for column in market_features if column.startswith("STOCK_")]
market_features["COE_DATA_AVAILABLE"] = market_features[coe_feature_columns].notna().any(axis=1).astype("int8")
market_features["STOCK_DATA_AVAILABLE"] = market_features[stock_feature_columns].notna().any(axis=1).astype("int8")

display(pd.DataFrame({
    "first_month": [market_features["RENT_DATE"].min()],
    "last_month": [market_features["RENT_DATE"].max()],
    "months": [len(market_features)],
    "coe_features": [len(coe_feature_columns)],
    "stock_features": [len(stock_feature_columns)],
}))

,first_month,last_month,months,coe_features,stock_features
0,2021-01-01,2026-08-01,68,20,7


## 7. Build HDB + conservative MRT transaction features

Only HDB block fields and the pre-2021 MRT proxy enter the final auxiliary merge. School, mall, COE, and stock features remain separate ablation candidates.

`ROW_ID` preserves the source relationship after core exact-deduplication. It is a join key, not a model feature.


In [10]:
hdb_mrt_model_columns = [
    "HDB_LATITUDE", "HDB_LONGITUDE", "HDB_MAX_FLOOR", "HDB_YEAR_COMPLETED",
    "HDB_MAX_FLOOR_INVALID", "HDB_YEAR_COMPLETED_INVALID", "HDB_POSTAL_SECTOR",
    "HDB_SUBZONE", "HDB_PLANNING_AREA", "HDB_REGION",
    "DIST_TO_NEAREST_PRE2021_MRT_KM",
    "PRE2021_MRT_COUNT_WITHIN_500M", "PRE2021_MRT_COUNT_WITHIN_1KM",
]
categorical_hdb_mrt_features = [
    "HDB_POSTAL_SECTOR", "HDB_SUBZONE", "HDB_PLANNING_AREA", "HDB_REGION",
]


def build_transaction_hdb_mrt_features(rental_data):
    keys = normalize_columns(rental_data[ADDRESS_KEYS], ADDRESS_KEYS)
    keys.insert(0, "ROW_ID", np.arange(len(keys), dtype=np.int64))
    keys["RENT_DATE"] = parse_rent_month(rental_data["RENT_APPROVAL_DATE"])

    feature_table = keys.merge(
        block_features[ADDRESS_KEYS + hdb_mrt_model_columns],
        on=ADDRESS_KEYS,
        how="left",
        validate="many_to_one",
        indicator="_HDB_MERGE",
    )
    feature_table["HDB_MATCHED"] = feature_table.pop("_HDB_MERGE").eq("both").astype("int8")

    building_age = feature_table["RENT_DATE"].dt.year - feature_table["HDB_YEAR_COMPLETED"]
    invalid_building_age = building_age.lt(0) | building_age.gt(100)
    feature_table["HDB_BUILDING_AGE_INVALID"] = invalid_building_age.fillna(False).astype("int8")
    feature_table["HDB_BUILDING_AGE_AT_RENT"] = building_age.mask(invalid_building_age).astype("float32")
    feature_table = feature_table.drop(columns=ADDRESS_KEYS)

    for column in categorical_hdb_mrt_features:
        feature_table[column] = feature_table[column].astype("string").fillna("__missing__")
    return feature_table.sort_values("ROW_ID").reset_index(drop=True)


train_hdb_mrt_features = build_transaction_hdb_mrt_features(train_raw)
test_hdb_mrt_features = build_transaction_hdb_mrt_features(test_raw)

train_join_rate = float(train_hdb_mrt_features["HDB_MATCHED"].mean())
test_join_rate = float(test_hdb_mrt_features["HDB_MATCHED"].mean())

assert len(train_hdb_mrt_features) == len(train_raw)
assert len(test_hdb_mrt_features) == len(test_raw)
assert train_hdb_mrt_features.columns.tolist() == test_hdb_mrt_features.columns.tolist()
assert train_hdb_mrt_features["ROW_ID"].tolist() == list(range(len(train_raw)))
assert test_hdb_mrt_features["ROW_ID"].tolist() == list(range(len(test_raw)))
assert min(train_join_rate, test_join_rate) >= 0.99
assert "MONTHLY_RENT" not in train_hdb_mrt_features.columns

display(pd.DataFrame({
    "rows": [len(train_hdb_mrt_features), len(test_hdb_mrt_features)],
    "columns": [train_hdb_mrt_features.shape[1], test_hdb_mrt_features.shape[1]],
    "HDB_join_rate": [train_join_rate, test_join_rate],
}, index=["train_hdb_mrt_features", "test_hdb_mrt_features"]))

,rows,columns,HDB_join_rate
train_hdb_mrt_features,150000,18,1.0000
test_hdb_mrt_features,50000,18,1.0000


## 8. Join, audit, and collapse duplicate feature vectors

The HDB/MRT matrix joins to core rows by original `ROW_ID`. The merged training table is then audited across every predictor, including the rental month. Rows with identical processed feature vectors are collapsed, and `MONTHLY_RENT` is replaced by the group mean. This prevents common vectors from receiving extra weight while preserving the squared-error-optimal target for each vector.

All test rows remain in original ID order. Their duplicate vectors legitimately receive the same model prediction but still produce separate submission rows.


In [11]:
EXISTING_AUXILIARY_PREFIXES = (
    "HDB_", "DIST_TO_NEAREST_", "MRT_COUNT_", "PRE2021_MRT_COUNT_",
    "MALL_COUNT_", "PRIMARY_SCHOOL_COUNT_", "COE_", "STOCK_", "BUILDING_AGE_",
)


def existing_auxiliary_columns(frame):
    return [column for column in frame.columns if column.startswith(EXISTING_AUXILIARY_PREFIXES)]


train_columns_to_remove = existing_auxiliary_columns(train_processed)
test_columns_to_remove = existing_auxiliary_columns(test_processed)
assert set(train_columns_to_remove) == set(test_columns_to_remove)

train_core_processed = train_processed.drop(columns=train_columns_to_remove)
test_core_processed = test_processed.drop(columns=test_columns_to_remove)

train_aux_for_merge = train_hdb_mrt_features.drop(columns="RENT_DATE")
test_aux_for_merge = test_hdb_mrt_features.drop(columns="RENT_DATE")

train_preprocessed_hdb_mrt = train_core_processed.merge(
    train_aux_for_merge, on="ROW_ID", how="left", validate="one_to_one"
)
test_preprocessed_hdb_mrt = test_core_processed.merge(
    test_aux_for_merge, on="ROW_ID", how="left", validate="one_to_one"
)

target = processed_metadata.get("target", "MONTHLY_RENT")
test_column_order = test_preprocessed_hdb_mrt.columns.tolist()
train_preprocessed_hdb_mrt = train_preprocessed_hdb_mrt[test_column_order + [target]]

feature_duplicate_subset = [column for column in test_column_order if column != "ROW_ID"]
precollapse_exact_duplicates = int(
    train_preprocessed_hdb_mrt.drop(columns="ROW_ID").duplicated().sum()
)
precollapse_feature_duplicates = int(
    train_preprocessed_hdb_mrt.duplicated(feature_duplicate_subset).sum()
)

groups = train_preprocessed_hdb_mrt.groupby(
    feature_duplicate_subset, dropna=False, sort=False, observed=True
)
group_sizes = groups.size()
collapsed_train = groups.agg(
    ROW_ID=("ROW_ID", "min"),
    MONTHLY_RENT=(target, "mean"),
).reset_index()
train_preprocessed_hdb_mrt = (
    collapsed_train[test_column_order + [target]]
    .sort_values("ROW_ID")
    .reset_index(drop=True)
)

duplicate_audit = pd.DataFrame({
    "rows": [len(train_core_processed), len(train_preprocessed_hdb_mrt)],
    "exact_duplicates": [precollapse_exact_duplicates, int(train_preprocessed_hdb_mrt.drop(columns="ROW_ID").duplicated().sum())],
    "duplicate_feature_rows": [precollapse_feature_duplicates, int(train_preprocessed_hdb_mrt.duplicated(feature_duplicate_subset).sum())],
    "largest_group": [int(group_sizes.max()), 1],
}, index=["before_feature_collapse", "after_feature_collapse"])
display(duplicate_audit)

assert len(test_preprocessed_hdb_mrt) == len(test_processed)
assert train_preprocessed_hdb_mrt.drop(columns=target).columns.tolist() == test_preprocessed_hdb_mrt.columns.tolist()
assert train_preprocessed_hdb_mrt["ROW_ID"].is_unique
assert test_preprocessed_hdb_mrt["ROW_ID"].tolist() == test_processed["ROW_ID"].tolist()
assert train_preprocessed_hdb_mrt.duplicated(feature_duplicate_subset).sum() == 0
assert not any(column.endswith(("_x", "_y")) for column in train_preprocessed_hdb_mrt.columns)
assert not any(column.startswith(("COE_", "STOCK_")) for column in train_preprocessed_hdb_mrt.columns)
assert "DIST_TO_NEAREST_PRE2021_MRT_KM" in train_preprocessed_hdb_mrt.columns
assert "DIST_TO_NEAREST_OPEN_MRT_KM" not in train_preprocessed_hdb_mrt.columns
assert "DIST_TO_NEAREST_MALL_KM" not in train_preprocessed_hdb_mrt.columns

display(pd.DataFrame({
    "rows": [len(train_preprocessed_hdb_mrt), len(test_preprocessed_hdb_mrt)],
    "columns": [train_preprocessed_hdb_mrt.shape[1], test_preprocessed_hdb_mrt.shape[1]],
    "feature_groups_collapsed": [precollapse_feature_duplicates, 0],
    "added_HDB_MRT_columns": [len(train_aux_for_merge.columns) - 1, len(test_aux_for_merge.columns) - 1],
}, index=["train_preprocessed_hdb_mrt", "test_preprocessed_hdb_mrt"]))


,rows,exact_duplicates,duplicate_feature_rows,largest_group
before_feature_collapse,149359,0,21333,7
after_feature_collapse,128026,0,0,1


,rows,columns,feature_groups_collapsed,added_HDB_MRT_columns
train_preprocessed_hdb_mrt,128026,38,21333,16
test_preprocessed_hdb_mrt,50000,37,0,16


## 9. Save HDB + MRT outputs

The saved HDB/MRT training table has one row per unique processed feature vector. The metadata records the collapse policy and before/after duplicate counts. Test and row-level auxiliary matrices remain uncollapsed for ID-level auditing and submission alignment.


In [12]:
final_non_feature_columns = ["ROW_ID", "RENT_DATE"]
final_model_features = [
    column for column in test_preprocessed_hdb_mrt.columns
    if column not in final_non_feature_columns
]
base_categorical_features = [
    column for column in processed_metadata.get("categorical_features", [])
    if column not in train_columns_to_remove
]
final_categorical_features = list(dict.fromkeys(
    base_categorical_features + categorical_hdb_mrt_features
))

metadata = {
    "target": target,
    "row_id": "ROW_ID",
    "date_column": "RENT_DATE",
    "source_processed_files": ["core/train_preprocessed.csv", "core/test_preprocessed.csv"],
    "output_files": ["hdb_mrt/train_preprocessed.csv", "hdb_mrt/test_preprocessed.csv"],
    "hdb_join_rate": {"train": train_join_rate, "test": test_join_rate},
    "added_hdb_mrt_features": [
        column for column in train_hdb_mrt_features.columns
        if column not in ["ROW_ID", "RENT_DATE"]
    ],
    "model_features": final_model_features,
    "categorical_features": final_categorical_features,
    "excluded_by_auxiliary_eda": ["schools", "shopping_malls", "coe", "stocks"],
    "mrt_time_policy": {
        "snapshot_status_not_used_alone": True,
        "proxy": "fixed conservative pre-2021 MRT network",
        "included_line_prefixes": sorted(PRE2021_MRT_PREFIXES),
        "excluded_later_codes": sorted(POST2021_OR_FUTURE_CODES),
    },
    "duplicate_policy": {
        "feature_columns": feature_duplicate_subset,
        "precollapse_exact_duplicates": precollapse_exact_duplicates,
        "precollapse_feature_duplicates": precollapse_feature_duplicates,
        "training_target_aggregation": "mean",
        "largest_duplicate_group": int(group_sizes.max()),
        "final_duplicate_feature_rows": 0,
        "test_rows_collapsed": False,
    },
}

if SAVE_OUTPUTS:
    AUXILIARY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    HDB_MRT_DIR.mkdir(parents=True, exist_ok=True)
    block_features.to_csv(AUXILIARY_OUTPUT_DIR / "hdb_spatial_features.csv", index=False)
    market_features.to_csv(AUXILIARY_OUTPUT_DIR / "market_features_lag1.csv", index=False, date_format="%Y-%m-%d")
    train_hdb_mrt_features.to_csv(AUXILIARY_OUTPUT_DIR / "train_hdb_mrt_features.csv", index=False, date_format="%Y-%m-%d")
    test_hdb_mrt_features.to_csv(AUXILIARY_OUTPUT_DIR / "test_hdb_mrt_features.csv", index=False, date_format="%Y-%m-%d")
    train_preprocessed_hdb_mrt.to_csv(HDB_MRT_DIR / "train_preprocessed.csv", index=False)
    test_preprocessed_hdb_mrt.to_csv(HDB_MRT_DIR / "test_preprocessed.csv", index=False)
    with (HDB_MRT_DIR / "preprocessing_metadata.json").open("w", encoding="utf-8") as handle:
        json.dump(metadata, handle, ensure_ascii=False, indent=2)

    saved_paths = [
        AUXILIARY_OUTPUT_DIR / "hdb_spatial_features.csv",
        AUXILIARY_OUTPUT_DIR / "market_features_lag1.csv",
        AUXILIARY_OUTPUT_DIR / "train_hdb_mrt_features.csv",
        AUXILIARY_OUTPUT_DIR / "test_hdb_mrt_features.csv",
        HDB_MRT_DIR / "preprocessing_metadata.json",
        HDB_MRT_DIR / "train_preprocessed.csv",
        HDB_MRT_DIR / "test_preprocessed.csv",
    ]
    print("Saved files:")
    for output_path in saved_paths:
        print(f"- {output_path.relative_to(ROOT)}: {output_path.stat().st_size / 1_048_576:.2f} MiB")


Saved files:
- data/processed/auxiliary/hdb_spatial_features.csv: 1.56 MiB
- data/processed/auxiliary/market_features_lag1.csv: 0.02 MiB
- data/processed/auxiliary/train_hdb_mrt_features.csv: 16.69 MiB
- data/processed/auxiliary/test_hdb_mrt_features.csv: 5.54 MiB
- data/processed/hdb_mrt/preprocessing_metadata.json: 0.00 MiB
- data/processed/hdb_mrt/train_preprocessed.csv: 39.11 MiB
- data/processed/hdb_mrt/test_preprocessed.csv: 14.94 MiB


## 10. Modeling handoff

- Compare `data/processed/core/` with `data/processed/hdb_mrt/` on the same chronological validation windows.
- Exclude `ROW_ID` and `RENT_DATE` from the model matrix; use `RENT_DATE` only to define time folds.
- Fit imputation, scaling, one-hot categories, and infrequent-category thresholds inside every training fold.
- The HDB/MRT training table contains one row per unique feature vector with mean rent as its target.
- The test table preserves all IDs, including duplicate vectors.
- Treat the pre-2021 MRT network as a conservative proxy. Replace it with month-specific station availability when opening dates become available.
